# Aargauer Bibliografie × swisscovery — Matching-Liste

Dieses Notebook erstellt eine Vorschlagsliste für neue deutschsprachige Wikipedia-Artikel:  
Personen/Musikgruppen aus dem [WikiProject Aargauer Bibliografie](https://www.wikidata.org/wiki/Wikidata:WikiProject_Aargauer_Bibliografie), die in swisscovery **mindestens zwei nicht-selbstverlegte Bücher** haben.

**Schritte:**
1. Setup & Imports
2. Wikidata-Fokusliste abrufen
3. swisscovery-Probe (3 Personen)
4. Vollabruf swisscovery (gecacht)
5. Matching anwenden
6. Export CSV + Markdown

## 1. Setup & Imports

In [ ]:
from __future__ import annotations
import json
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

from aargau_match.wikidata import fetch_focus_list
from aargau_match.swisscovery import search_by_gnd, search_by_name, BookRecord
from aargau_match.matching import build_person_result
from aargau_match.exporters import to_csv, to_markdown

DATA = Path('data')
DATA.mkdir(exist_ok=True)
(DATA / 'output').mkdir(exist_ok=True)

WIKIDATA_CSV = DATA / 'wikidata_persons.csv'
CACHE_FILE = DATA / 'swisscovery_hits.json'
OUT_CSV = DATA / 'output' / 'matching_liste.csv'
OUT_MD = DATA / 'output' / 'matching_liste.md'

## 2. Wikidata-Fokusliste abrufen

SPARQL gegen `query.wikidata.org`. ~2.000+ Personen (Stand 2026).

In [ ]:
if WIKIDATA_CSV.exists():
    persons = pd.read_csv(WIKIDATA_CSV, keep_default_na=False)
    print(f'Aus Cache geladen: {len(persons)} Personen')
else:
    persons = fetch_focus_list()
    persons.to_csv(WIKIDATA_CSV, index=False)
    print(f'Neu abgerufen: {len(persons)} Personen → {WIKIDATA_CSV}')

print(f'  davon mit GND-ID: {(persons.gnd != "").sum()}')
print(f'  davon mit dewiki-Artikel: {persons.has_dewiki.astype(bool).sum()}')
persons.head()

## 3. swisscovery-Probe (3 Personen)

Sanity-Check, dass GND- und Name-Suche funktionieren.

In [ ]:
probe_qids = ['Q123127']  # Klaus Merz
for qid in probe_qids:
    row = persons[persons.q_id == qid]
    if row.empty:
        continue
    p = row.iloc[0].to_dict()
    print(f"--- {p['label']} (GND={p['gnd'] or '–'}) ---")
    hits = search_by_gnd(str(p['gnd'])) if p['gnd'] else search_by_name(p['family'], p['given'])
    conf = 'gnd' if p['gnd'] else 'name-fuzzy'
    res = build_person_result(p, hits, confidence=conf)
    print(f'  raw hits: {len(hits)} | nach Creator-Filter+Dedup: {res.books_total}')
    print(f'  non-selfpub: {res.books_non_selfpub} | selfpub: {res.books_selfpub} | qualifies: {res.qualifies}')
    print(f'  Verlage (erste): {res.publishers[:200]}')

## 4. Vollabruf swisscovery

Für jede Person wird gesucht — primär per GND, sonst per Name. Treffer werden lokal in `data/swisscovery_hits.json` gecacht; ein Re-Run überspringt bereits abgerufene Personen.

Dauer: ca. 30–60 min beim ersten Lauf (abhängig von Rate-Limits & Anzahl Treffer).

In [ ]:
if CACHE_FILE.exists():
    cache = json.loads(CACHE_FILE.read_text())
else:
    cache = {}
print(f'Cache-Einträge: {len(cache)}')

todo = [r for _, r in persons.iterrows() if r.q_id not in cache]
print(f'Noch abzurufen: {len(todo)} / {len(persons)}')

FLUSH_EVERY = 50
for i, row in enumerate(tqdm(todo)):
    p = row.to_dict()
    qid = p['q_id']
    try:
        if p.get('gnd'):
            hits = search_by_gnd(str(p['gnd']))
            conf = 'gnd'
        elif p.get('family') or p.get('given'):
            hits = search_by_name(p.get('family',''), p.get('given',''))
            conf = 'name-fuzzy'
        else:
            hits, conf = [], 'none'
        cache[qid] = {
            'confidence': conf,
            'hits': [h.to_dict() for h in hits],
        }
    except Exception as exc:
        cache[qid] = {'confidence': 'error', 'hits': [], 'error': str(exc)}
    if (i + 1) % FLUSH_EVERY == 0:
        CACHE_FILE.write_text(json.dumps(cache, ensure_ascii=False))

CACHE_FILE.write_text(json.dumps(cache, ensure_ascii=False))
print(f'Fertig. Cache: {len(cache)} Einträge.')

## 5. Matching anwenden

Pro Person: Creator-vs-Subject-Filter, Selbstverlags-Heuristik, Qualifikation prüfen.

In [ ]:
results = []
for _, row in persons.iterrows():
    p = row.to_dict()
    entry = cache.get(p['q_id'], {'confidence': 'none', 'hits': []})
    hits = [BookRecord(**h) for h in entry['hits']]
    res = build_person_result(p, hits, confidence=entry.get('confidence', 'none'))
    results.append(res.as_row())
results_df = pd.DataFrame(results)
print(f'Gesamt: {len(results_df)}')
print(f'  qualifies=True: {results_df.qualifies.sum()}')
print(f'  davon ohne dewiki-Artikel: {((results_df.qualifies) & (~results_df.has_dewiki)).sum()}')
print(f'  davon requires_review: {((results_df.qualifies) & (results_df.requires_review)).sum()}')

fig, ax = plt.subplots(figsize=(8, 3))
results_df.books_non_selfpub.clip(upper=20).hist(bins=range(0, 22), ax=ax)
ax.set_xlabel('Anzahl nicht-selbstverlegte Bücher (≥20 = 20+)')
ax.set_ylabel('Personen')
ax.axvline(2, color='red', linestyle='--', label='Schwellwert (≥2)')
ax.legend()
plt.show()

## 6. Export CSV + Markdown

In [ ]:
csv_path = to_csv(results_df, OUT_CSV)
md_path = to_markdown(results_df, OUT_MD)
print(f'CSV: {csv_path}')
print(f'Markdown: {md_path}')

# Top-10-Kandidat/innen ohne dewiki-Artikel
top = results_df[(results_df.qualifies) & (~results_df.has_dewiki)].sort_values('books_non_selfpub', ascending=False)
top[['name','gnd','books_non_selfpub','confidence','requires_review','wikidata_url']].head(10)